# 五項困難：第二輪可重現實驗
本輪已測得的條件村落模型未通過兩種子門檻；不保證Kaggle0.680。此Notebook會重新訓練，並將結果寫到新目錄，保留既有成果。

請從GitHub分支 `research-0680-20261024` 的 Code → Download ZIP 下載完整儲存庫，再於下面上傳。私人儲存庫先在瀏覽器登入下載，不需要Token。如果使用 `Kaggle_Phase2_complete.zip` 交付包，請同時上傳原始 `train.csv`、`test.csv`、`sample_submission.csv`。


In [ ]:
from pathlib import Path
import os,sys,subprocess,json,zipfile,time,shutil
base=Path.cwd()
matches=[p.parent for p in base.rglob('train.csv') if (p.parent/'research_0680'/'src').is_dir() and '.venvs' not in p.parts]
if not matches:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError('請在既有儲存庫目錄執行，或在 Colab 上傳完整儲存庫 ZIP。')
    uploaded=files.upload()
    target=base/'earthquake_repository';target.mkdir(exist_ok=True)
    for name in uploaded:
        if name.endswith('.zip'):
            with zipfile.ZipFile(name) as archive:
                for item in archive.infolist():
                    destination=(target/item.filename).resolve()
                    if not destination.is_relative_to(target.resolve()):
                        raise ValueError('拒絕 ZIP 路徑越界')
                archive.extractall(target)
    for project in target.rglob('research_0680'):
        if not (project/'src').is_dir():continue
        for name in ['train.csv','test.csv','sample_submission.csv']:
            if name in uploaded and not (project.parent/name).exists():
                shutil.copyfile(base/name,project.parent/name)
    matches=[p.parent for p in target.rglob('train.csv') if (p.parent/'research_0680'/'src').is_dir()]
if not matches:
    raise RuntimeError('找不到同時含 train.csv 與 research_0680/src 的完整儲存庫。請使用 GitHub Code → Download ZIP。')
repo=matches[0].resolve()
assert all((repo/f).is_file() for f in ['train.csv','test.csv','sample_submission.csv'])
print('Repository:',repo)


In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','-r',str(repo/'research_0680/configs/requirements.txt')])
for name in ['OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ[name]='1'
run_dir=repo/'research_0680'/'phase2_colab_runs'/time.strftime('%Y%m%d_%H%M%S')
run_dir.mkdir(parents=True,exist_ok=False)
os.environ['RESEARCH0680_RESULTS']=str(run_dir/'results')
os.environ['RESEARCH0680_REPORTS']=str(run_dir/'reports')
os.environ['RESEARCH0680_PHASE2_RESULTS']=str(run_dir/'phase2_results')
os.environ['RESEARCH0680_PHASE2_REPORTS']=str(run_dir/'phase2_reports')
os.environ['MPLCONFIGDIR']=str(run_dir/'matplotlib_cache')
print('Fresh output directory:',run_dir)


## 完整重跑
執行說法核對、九項開發消融、兩種子外層與內層選模、量測重建及提交門檻。新輸出目錄沒有參考OOF時會重新計算固定參考。CPU可能需數分鐘至數十分鐘。
所有最終λ與混合權重僅由外層訓練資料內層選擇；開發集最高值不是Kaggle分數。使用者0.679原程式目前不可原樣重現，本參考OOF不得冒稱其OOF。


In [ ]:
subprocess.check_call([sys.executable,'-B','-m','research_0680.phase2.src.run_all'],cwd=repo,env=os.environ.copy())


In [ ]:
print((run_dir/'phase2_results/nested_comparison.csv').read_text())
print((run_dir/'phase2_results/submission_gate.json').read_text())
print((run_dir/'phase2_reports/final_report.md').read_text())


## 下載
只有兩種子都通過ΔBA>.001且paired95%CI下界>0，才建立新候選。既有結果未通過，所以沒有新提交。資料、程式、CSV、指標與紀錄會包含在下列重跑ZIP。


In [ ]:
bundle=base/'Kaggle_Phase2_reproduction_results.zip'
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in run_dir.rglob('*'):
        if p.is_file() and 'matplotlib_cache' not in p.parts:
            z.write(p,p.relative_to(run_dir))
print(bundle,bundle.stat().st_size)
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    print('本機結果位於上面列出的 ZIP 路徑。')
